# ⚡ CircuitAI — Colab GPU Backend
**Step 1**: Runtime → Change runtime type → T4 or L4 GPU

**Step 2**: Run cells in order.

**Step 3**: Wait for `✅ All models ready` in the logs, then paste the `trycloudflare.com` URL into the frontend.

In [ ]:
# ── Cell 0: Mount Drive & Set Working Directory ─────────────────
import sys, os
from google.colab import drive

drive.mount('/content/drive')

# ⚠️ Update this path if your folder is in a different location
BACKEND_DIR = '/content/drive/MyDrive/knowledge_graph/backend'

os.chdir(BACKEND_DIR)
if BACKEND_DIR not in sys.path:
    sys.path.insert(0, BACKEND_DIR)

print(f'Working directory: {os.getcwd()}')
print(f'Files: {os.listdir()}')

In [ ]:
# ── Cell 1: Install Cloudflared ────────────────────────────────
!curl -fsSL https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb -o cloudflared.deb
!dpkg -i cloudflared.deb
print('Cloudflared installed ✓')

In [ ]:
# ── Cell 2: Install Python dependencies ────────────────────────
!pip install -r requirements.txt -q
print('Python deps installed ✓')

In [ ]:
# ── Cell 3: Start FastAPI server + Cloudflare Tunnel ───────────
import subprocess, threading, time, uvicorn, os

# ── Kill any existing server on port 8000 ─────────────────────
# This handles the 'address already in use' error on re-runs
os.system("fuser -k 8000/tcp 2>/dev/null; sleep 1")
print('Port 8000 cleared ✓')

from main import app

def start_server():
    uvicorn.run(app, host='0.0.0.0', port=8000, log_level='info')

server_thread = threading.Thread(target=start_server, daemon=True)
server_thread.start()

# ── Wait for models to finish loading before starting tunnel ───
# Qwen2.5-VL-3B on T4 takes ~90s to load and quantize
print('⏳ Loading models into GPU... (this takes ~90 seconds)')
print('   Watch the logs above for: [startup] ✅ All models ready')

# Poll until the server is actually accepting connections
import urllib.request
for i in range(90):           # wait up to 90 seconds
    time.sleep(2)
    try:
        urllib.request.urlopen('http://localhost:8000/health', timeout=2)
        print(f'\n✅ Server is up and healthy after {(i+1)*2}s')
        break
    except Exception:
        print(f'  [{(i+1)*2}s] Still loading...', end='\r')
else:
    print('\n⚠️  Server did not respond in time — check logs above for errors.')

# ── Start Cloudflare Tunnel ────────────────────────────────────
print('\nStarting Cloudflare tunnel...')
p = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://localhost:8000'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

for line in p.stdout:
    print(line, end='')
    if 'trycloudflare.com' in line:
        tokens = line.split()
        public_url = next((t for t in tokens if 'trycloudflare.com' in t), None)
        if public_url:
            print('\n' + '='*60)
            print(f'  🔥 PUBLIC URL: {public_url}')
            print('  Paste into CircuitAI frontend > Connection field')
            print('='*60)
        break